In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan untuk menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-addon-utils~=0.3.0
```
</details>

Paket utilitas addon Qiskit adalah kumpulan fungsionalitas untuk melengkapi workflow yang melibatkan satu atau lebih addon Qiskit. Misalnya, paket ini berisi fungsi-fungsi untuk membuat Hamiltonian, menghasilkan Circuit evolusi waktu Trotter, serta memotong dan menggabungkan Circuit kuantum.

## Instalasi

Ada dua cara untuk menginstal utilitas addon Qiskit: dari PyPI dan membangun dari sumber. Disarankan untuk menginstal paket-paket ini di [virtual environment](https://docs.python.org/3.10/tutorial/venv.html) agar dependensi paket tetap terpisah.

### Instal dari PyPI

Cara paling mudah untuk menginstal paket utilitas addon Qiskit adalah melalui PyPI.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import QuantumCircuit
from qiskit_addon_utils.coloring import auto_color_edges
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth
from collections import defaultdict

backend = FakeSherbrooke()
coupling_map = backend.coupling_map

# Create naive circuit
circuit = QuantumCircuit(backend.num_qubits)
for edge in coupling_map.graph.edge_list():
    circuit.cz(edge[0], edge[1])


# Color the edges of the coupling map
coloring = auto_color_edges(coupling_map)
circuit_with_coloring = QuantumCircuit(backend.num_qubits)

# Make a reverse coloring dict in order to make the circuit
color_to_edge = defaultdict(list)
for edge, color in coloring.items():
    color_to_edge[color].append(edge)

# Place edges in order of color
for edges in color_to_edge.values():
    for edge in edges:
        circuit_with_coloring.cz(edge[0], edge[1])

print(f"The circuit without using edge coloring has depth: {circuit.depth()}")
print(
    f"The circuit using edge coloring has depth: {circuit_with_coloring.depth()}"
)

The circuit without using edge coloring has depth: 37
The circuit using edge coloring has depth: 3


### Instal dari sumber
<details>
<summary>
Klik di sini untuk membaca cara menginstal paket ini secara manual.
</summary>

Jika kamu ingin berkontribusi pada paket ini atau ingin menginstalnya secara manual, clone dulu repositorinya:

In [2]:
import numpy as np
from qiskit import QuantumCircuit

num_qubits = 9
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg" alt="Output of the previous code cell" />

lalu instal paket melalui `pip`. Jika kamu berencana menjalankan tutorial yang ada di repositori paket, instal juga dependensi notebook-nya. Jika kamu berencana untuk mengembangkan di repositori tersebut, instal dependensi `dev`.

In [3]:
# Slice circuit into partitions of depth 1
slices = slice_by_depth(qc, 1)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/6d824d97-edc6-4c88-bcfa-428731393f83-0.svg" alt="Output of the previous code cell" />

</details>

## Mulai menggunakan utilitas
Ada beberapa modul dalam paket `qiskit-addon-utils`, termasuk satu untuk pembuatan masalah dalam simulasi sistem kuantum, pewarnaan graf untuk penempatan Gate yang lebih efisien dalam Circuit kuantum, dan pemotongan Circuit, yang dapat membantu dengan [propagasi balik operator](/guides/qiskit-addons-obp). Bagian-bagian berikut merangkum setiap modul. [Dokumentasi API](https://docs.quantum.ibm.com/api/qiskit-addon-utils) paket ini juga berisi informasi yang berguna.

### Pembuatan masalah
Isi dari modul [`qiskit_addon_utils.problem_generators`](../api/qiskit-addon-utils/problem-generators) meliputi:

- Fungsi [`generate_xyz_hamiltonian()`](../api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian), yang menghasilkan representasi `SparsePauliOp` yang sadar konektivitas dari model XYZ tipe Ising:

$$H = \sum_{(j,k)\in E} \left(J_x X_jX_k + J_yY_jY_k + J_zZ_jZ_k\right) + \sum_{j\in V} \left(h_x X_j + h_y Y_j + h_z Z_j\right) $$
- Fungsi [`generate_time_evolution_circuit()`](../api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit), yang membangun Circuit yang memodelkan evolusi waktu dari operator yang diberikan.
- Tiga objek [`PauliOrderStrategy`](../api/qiskit-addon-utils/problem-generators#pauliorderstrategy) berbeda untuk menghitung antara berbagai urutan string Pauli. Ini sangat berguna saat digunakan bersama pewarnaan graf dan bisa dipakai sebagai argumen di fungsi `generate_xyz_hamiltonian()` maupun `generate_time_evolution_circuit()`.

### Pewarnaan graf
Modul [`qiskit_addon_utils.coloring`](../api/qiskit-addon-utils/coloring) digunakan untuk mewarnai tepi-tepi dalam coupling map dan menggunakan pewarnaan ini untuk menempatkan Gate dalam Circuit kuantum secara lebih efisien. Tujuan dari coupling map tepi-berwarna ini adalah untuk menemukan sekumpulan warna tepi sehingga tidak ada dua tepi dengan warna yang sama yang berbagi node yang sama. Untuk QPU, ini berarti Gate di sepanjang tepi dengan warna yang sama (koneksi Qubit) bisa dijalankan secara bersamaan sehingga Circuit akan dieksekusi lebih cepat.

Sebagai contoh singkat, kamu bisa menggunakan fungsi [`auto_color_edges()`](../api/qiskit-addon-utils/coloring#auto_color_edges) untuk menghasilkan pewarnaan tepi bagi Circuit naif yang mengeksekusi `CZGate` di setiap koneksi Qubit. Cuplikan kode di bawah ini menggunakan coupling map Backend `FakeSherbrooke`, membuat Circuit naif ini, lalu menggunakan fungsi `auto_color_edges()` untuk membuat Circuit ekivalen yang lebih efisien.

In [4]:
from qiskit_addon_utils.slicing import slice_by_gate_types

slices = slice_by_gate_types(qc)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d011c56e-d741-491a-8867-54952cb7b757-0.svg" alt="Output of the previous code cell" />

If your workload is designed to exploit the physical qubit connectivity of the QPU it will be run on, you can create slices based on edge coloring. The code snippet below will assign a three-coloring to circuit edges and slice the circuit with respect to the edge coloring. (Note: this only affects non-local gates. Single-qubit gates will be sliced by gate type).

In [5]:
from qiskit_addon_utils.slicing import slice_by_coloring

# Assign a color to each set of connected qubits
coloring = {}
for i in range(num_qubits - 1):
    coloring[(i, i + 1)] = i % 3
coloring[(num_qubits - 1, 0)] = 2

# Create a circuit with operations added in order of color
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
edges = [
    edge for color in range(3) for edge in coloring if coloring[edge] == color
]
for edge in edges:
    qc.cx(edge[0], edge[1])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))

# Create slices by edge color
slices = slice_by_coloring(qc, coloring=coloring)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg" alt="Output of the previous code cell" />

### Pemotongan
Terakhir, modul [`qiskit-addon-utils.slicing`](../api/qiskit-addon-utils/slicing) berisi fungsi-fungsi dan pass Transpiler untuk bekerja dengan pembuatan "irisan" Circuit, partisi mirip-waktu dari [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit) yang mencakup semua Qubit. Irisan-irisan ini terutama digunakan untuk [propagasi balik operator](/guides/qiskit-addons-obp-get-started). Empat cara utama sebuah Circuit bisa dipotong adalah berdasarkan jenis Gate, kedalaman, pewarnaan, atau instruksi [`Barrier`](../api/qiskit/circuit#barrier). Output dari fungsi-fungsi pemotongan ini menghasilkan daftar objek [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit). Circuit yang sudah dipotong juga bisa digabungkan kembali menggunakan fungsi [`combine_slices()`](../api/qiskit-addon-utils/slicing#combine_slices). Baca [referensi API](../api/qiskit-addon-utils/slicing) modul ini untuk informasi lebih lanjut.

Di bawah ini adalah beberapa contoh cara membuat irisan-irisan ini menggunakan Circuit berikut:

In [6]:
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qc.barrier()
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.barrier()
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/5040a678-9d5f-4c8b-8a92-7de04c3fcfda-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg)

Dalam kasus di mana tidak ada cara yang jelas untuk memanfaatkan struktur Circuit untuk propagasi balik operator, kamu bisa mempartisi Circuit menjadi irisan dengan kedalaman tertentu.

In [7]:
from qiskit_addon_utils.slicing import slice_by_barriers

slices = slice_by_barriers(qc)
slices[0].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/b1106c2f-711c-4d30-b91a-ea05f47598d8-0.svg" alt="Output of the previous code cell" />

In [8]:
slices[1].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/1afd2111-dd88-4e20-a137-f0c975e9d1bb-0.svg" alt="Output of the previous code cell" />

In [9]:
slices[2].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/360ab773-df81-4975-bb19-cd5f34e69b29-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg)

Jika kamu punya strategi pemotongan kustom, kamu bisa menempatkan Barrier dalam Circuit untuk menandai di mana Circuit harus dipotong dan menggunakan fungsi [`slice_by_barriers`](../api/qiskit-addon-utils/slicing#slice_by_barriers).